# Import libraries

In [1]:
import keras, subprocess, gc, os, datetime, numpy as np
import tensorflow as tf
from keras import layers, datasets, layers, models, losses

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppresses INFO and WARNING messages

# Make TF only allocate as much memory as required for the model during training and inference
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

# Check if GPU is available
print(tf.config.list_physical_devices('GPU'))

2024-12-01 13:19:25.579546: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-01 13:19:25.759619: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-01 13:19:25.759895: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-01 13:19:25.781761: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-01 13:19:25.821312: I tensorflow/core/platform/cpu_feature_guar

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-12-01 13:19:29.244042: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 13:19:29.674271: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 13:19:29.674327: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


# Support Functions

In [2]:
global x_val, y_val, x_train, y_train
def create_subset(number_of_inputs):
    global x_test_sub, y_test_sub, x_train_sub, y_train_sub
    x_test_sub = x_test[-number_of_inputs:,:,:,:]
    y_test_sub = y_test[-number_of_inputs:]
    x_train_sub = x_train[:-number_of_inputs,:,:,:]
    y_train_sub = y_train[:-number_of_inputs]

def load_model_weights():
    global model_path, model
    # Creates model folder
    if not os.path.exists(model_path):
        os.mkdir(model_path)
        
    model_path = model_path + '/' + model_path
    
    # Load weghts if file exists
    if os.path.exists(model_path+'.weights.h5'):
        model.load_weights(model_path+'.weights.h5')

# Datasets and Preprocessing

### MNIST

In [3]:
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()

# Add padding 2x2 and trnasform input into floating point
x_train = tf.pad(x_train, [[0, 0], [2, 2], [2, 2]]) / 255
x_test = tf.pad(x_test, [[0, 0], [2, 2], [2, 2]]) / 255
x_train = tf.expand_dims(x_train, axis=3, name=None)
x_test = tf.expand_dims(x_test, axis=3, name=None)
print(x_test.shape)

(10000, 32, 32, 1)


2024-12-01 03:30:59.512329: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 03:30:59.512408: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 03:30:59.512424: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 03:31:00.600924: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 03:31:00.601044: I external/local_xla/xla/stream_executor

### CIFAR

### Fashion MNIST

In [7]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
# Add padding 2x2 and trnasform input into floating point
x_train = tf.pad(x_train, [[0, 0], [2, 2], [2, 2]]) / 255
x_test = tf.pad(x_test, [[0, 0], [2, 2], [2, 2]]) / 255
x_train = tf.expand_dims(x_train, axis=3, name=None)
x_test = tf.expand_dims(x_test, axis=3, name=None)
print(x_train.shape)

(60000, 32, 32, 1)


# Models

## LCNN

In [4]:
model_path='lcnn_fashion_og'

class LCNN(models.Model):
    def __init__(self, input_shape=(32, 32, 1), num_classes=10):
        super(LCNN, self).__init__()

        # Define the layers
        self.conv1 = layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=input_shape)
        self.conv2 = layers.Conv2D(32, kernel_size=(3, 3), activation='relu')
        self.maxpool1 = layers.MaxPooling2D(pool_size=(2, 2))

        self.conv3 = layers.Conv2D(64, kernel_size=(3, 3), activation='relu')
        self.conv4 = layers.Conv2D(64, kernel_size=(3, 3), activation='relu')
        self.maxpool2 = layers.MaxPooling2D(pool_size=(2, 2))

        self.flatten = layers.Flatten()
        self.dense1 = layers.Dense(128, activation='relu')
        
        self.output_layer = layers.Dense(num_classes, activation='sigmoid')

    def call(self, inputs):
        # Define the forward pass
        x = self.conv1(inputs)
        x = self.conv2(x)
        x = self.maxpool1(x)

        x = self.conv3(x)
        x = self.conv4(x)
        x = self.maxpool2(x)

        x = self.flatten(x)
        x = self.dense1(x)
        x = self.output_layer(x)
        return x
        

# Create the Lightweight CNN model
model = LCNN()

2024-12-01 01:29:35.101960: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 01:29:35.102975: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 01:29:35.103069: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 01:29:37.397896: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-12-01 01:29:37.398617: I external/local_xla/xla/stream_executor

## Tiny VGG

In [36]:
model_path='tinyvgg_fashion_og'

class TinyVGG(models.Model):
    def __init__(self, input_shape=(32, 32, 1), num_classes=10):
        super(TinyVGG, self).__init__()

        # Block 1
        self.conv1_1 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same', input_shape=input_shape)
        self.conv1_2 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same')
        self.maxpool1 = layers.MaxPooling2D(pool_size=(2, 2), strides=2)

        # Block 2
        self.conv2_1 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same')
        self.conv2_2 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same')
        self.maxpool2 = layers.MaxPooling2D(pool_size=(2, 2), strides=2)

        # Block 3
        self.conv3_1 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same')
        self.conv3_2 = layers.Conv2D(16, kernel_size=(3, 3), activation='relu', padding='same')
        self.maxpool3 = layers.MaxPooling2D(pool_size=(2, 2), strides=2)

        # Fully connected layers
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(256, activation='relu')
        self.fc2 = layers.Dense(128, activation='relu')
        self.fc3 = layers.Dense(64, activation='relu')
        self.fc4 = layers.Dense(num_classes, activation='softmax')

    def call(self, inputs):
        # Block 1
        x = self.conv1_1(inputs)
        x = self.conv1_2(x)
        x = self.maxpool1(x)

        # Block 2
        x = self.conv2_1(x)
        x = self.conv2_2(x)
        x = self.maxpool2(x)

        # Block 3
        x = self.conv3_1(x)
        x = self.conv3_2(x)
        x = self.maxpool3(x)

        # Fully connected layers
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return self.fc4(x)

model = TinyVGG()

## LeNet-5

### Lenet OG

In [ ]:
model_path='lenet5_mnist_og'

class Lenet(tf.keras.Model):

  def __init__(self):
    super().__init__()
    self.conv1 = self._make_conv_layer(6, 5, input_shape=[32, 32, 1])
    self.conv2 = self._make_conv_layer(16, 5)
    self.conv3 = self._make_conv_layer(120, 5, pooling=False, flatten=True)
    self.dense1 = layers.Dense(84, activation='tanh')
    self.dense2 = layers.Dense(10, activation='softmax')

  def call(self, inputs):
    x = self.conv1(inputs)
    x = self.conv2(x)
    x = self.conv3(x)
    x = self.dense1(x)
    x = self.dense2(x)

    return x

  def _make_conv_layer(self, filters, kernel_size, input_shape=None, pooling=True, flatten=False):
    l = keras.Sequential()
    if input_shape is not None:
      l.add(layers.Conv2D(filters, kernel_size, activation='tanh', input_shape=input_shape))
    else:
      l.add(layers.Conv2D(filters, kernel_size, activation='tanh'))
    if pooling:
      l.add(layers.AveragePooling2D(2))
      l.add(layers.Activation('sigmoid'))
    if flatten:
      l.add(layers.Flatten())
    return l

model = Lenet()

In [4]:
model_path = 'lenet5_mnist_og'

class LeNet5(tf.keras.Model):
    def __init__(self, num_classes=10):
        super(LeNet5, self).__init__()

        # First convolutional layer with input shape defined
        self.conv1 = layers.Conv2D(filters=6, kernel_size=(5, 5), activation='tanh', padding='same', input_shape=[32, 32, 3])
        self.pool1 = layers.AveragePooling2D(pool_size=(2, 2), strides=2)

        # Second convolutional layer
        self.conv2 = layers.Conv2D(filters=16, kernel_size=(5, 5), activation='tanh', padding='valid')
        self.pool2 = layers.AveragePooling2D(pool_size=(2, 2), strides=2)

        # Third convolutional layer
        self.conv3 = layers.Conv2D(filters=120, kernel_size=(5, 5), activation='tanh', padding='valid')

        # Flatten before Dense layers
        self.flatten = layers.Flatten()

        # First fully connected layer
        self.fc1 = layers.Dense(units=84, activation='tanh')

        # Output layer with softmax activation
        self.output_layer = layers.Dense(units=num_classes, activation='softmax')

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)

        x = self.flatten(x)

        x = self.fc1(x)

        return self.output_layer(x)

model = LeNet5()

# Model Analysis

In [8]:
model.build(x_train.shape)
load_model_weights()
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 32, 32, 3)]          0         []                            
                                                                                                  
 conv2d_33 (Conv2D)          (None, 32, 32, 16)           448       ['input_1[0][0]']             
                                                                                                  
 batch_normalization_49 (Ba  (None, 32, 32, 16)           64        ['conv2d_33[0][0]']           
 tchNormalization)                                                                                
                                                                                                  
 re_lu_17 (ReLU)             (None, 32, 32, 16)           0         ['batch_normalization_49[0

In [22]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), 
              loss=losses.sparse_categorical_crossentropy, 
              metrics=['accuracy'])
loss_rec, accuracy_rec = model.evaluate(x_test, y_test)

313/313 [==============================] - 2s 3ms/step - loss: 0.3405 - accuracy: 0.8898


# Train

In [6]:
epochs = 100
batch_size = 64
create_subset(number_of_inputs = 4000) # Creates a smaller validation set

In [7]:
checkpoint_path = model_path+".weights.h5"
checkpoint_dir = os.path.dirname(checkpoint_path)

# Create a callback that saves the model's weights
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path, save_weights_only=True, verbose=1)

# TensorBoard
#os.remove(logs)
log_dir = "_logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S") # Create a unique log directory
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
%load_ext tensorboard
# run tensorboard --logdir _logs/fit and open http://localhost:6006/ 

In [8]:
saved_history = []
# Saves the loss of the training
if os.path.exists(model_path+'_loss.npy'):  
    saved_history = np.load(model_path+'_loss.npy')
    print(f"Number of epochs already trained: {len(saved_history)}")

# Create a callback that saves the model's weights
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), 
              loss=losses.sparse_categorical_crossentropy, 
              metrics=['accuracy'])
history = model.fit(x_train, y_train, 
                    batch_size=batch_size, 
                    epochs=epochs, 
                    validation_split=0.1, 
                    callbacks=[cp_callback, tensorboard_callback, early_stopping])

# Record number of epochs trained
saved_history = np.append(saved_history, history.history['loss'])
np.save(model_path+'_loss.npy', saved_history)
print(f"Number of epochs trained: {len(saved_history)}")


Epoch 1/100


2024-12-01 03:32:06.253960: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
2024-12-01 03:32:07.433711: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f9260777ba0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-12-01 03:32:07.433753: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX 2000 Ada Generation Laptop GPU, Compute Capability 8.9
2024-12-01 03:32:07.440746: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1733020327.516918    2682 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


844/844 [==============================] - ETA: 0s - loss: 0.2741 - accuracy: 0.9200
Epoch 1: saving model to lenet5_mnist_og/lenet5_mnist_og.weights.h5
844/844 [==============================] - 10s 8ms/step - loss: 0.2741 - accuracy: 0.9200 - val_loss: 0.1178 - val_accuracy: 0.9688
Epoch 2/100
839/844 [============================>.] - ETA: 0s - loss: 0.1155 - accuracy: 0.9645
Epoch 2: saving model to lenet5_mnist_og/lenet5_mnist_og.weights.h5
844/844 [==============================] - 4s 5ms/step - loss: 0.1153 - accuracy: 0.9645 - val_loss: 0.0852 - val_accuracy: 0.9753
Epoch 3/100
840/844 [============================>.] - ETA: 0s - loss: 0.0779 - accuracy: 0.9762
Epoch 3: saving model to lenet5_mnist_og/lenet5_mnist_og.weights.h5
844/844 [==============================] - 5s 6ms/step - loss: 0.0778 - accuracy: 0.9762 - val_loss: 0.0670 - val_accuracy: 0.9788
Epoch 4/100
844/844 [==============================] - ETA: 0s - loss: 0.0586 - accuracy: 0.9818
Epoch 4: saving model to l

### Model sizing and saving

In [ ]:
def get_file_size(file_path):
    size = os.path.getsize(file_path)
    return size
def convert_bytes(size, unit=None):
    if unit == "KB":
        return print('File size: ' + str(round(size / 1024, 3)) + ' Kilobytes')
    elif unit == "MB":
        return print('File size: ' + str(round(size / (1024 * 1024), 3)) + ' Megabytes')
    else:
        return print('File size: ' + str(size) + ' bytes')

In [9]:
# This is not very nice: earlier version 2.11.0 says 
# Saving the model to HDF5 format requires the model to be a Functional model or a Sequential model. 
# It does not work for subclassed models, because such models are defined via the body of a Python method, which isn't safely serializable. 
# Consider saving to the Tensorflow SavedModel format (by setting save_format="tf") or using `save_weights`.
model.save(model_path+'.keras')

convert_bytes(get_file_size(model_path+".keras"), "KB")

NameError: name 'convert_bytes' is not defined

In [ ]:
# Delete model to free memory
#del model, history, callbacks
del x_train
gc.collect()

# Release GPU memory
tf.keras.backend.clear_session()

# Quantization and TFLite

In [10]:
# TensorFlow Lite needs a representative dataset in order to determine the correct quantization parameters
def representative_dataset():
  for data in tf.data.Dataset.from_tensor_slices((x_test)).batch(1).take(100):
    yield [tf.dtypes.cast(data, tf.float32)]
        
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8  # or tf.uint8
converter.inference_output_type = tf.int8  # or tf.uint8

# Convert the model
tflite_model = converter.convert()

# Save the quantized model
with open(model_path + '.tflite', 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: /tmp/tmpxu_ezpke/assets


INFO:tensorflow:Assets written to: /tmp/tmpxu_ezpke/assets
/home/leo/miniconda3/envs/tf/lib/python3.9/site-packages/tensorflow/lite/python/convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2024-12-01 03:36:05.155181: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2024-12-01 03:36:05.155229: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2024-12-01 03:36:05.155462: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpxu_ezpke
2024-12-01 03:36:05.156553: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2024-12-01 03:36:05.156564: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpxu_ezpke
2024-12-01 03:36:05.158874: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization p

In [11]:
input_file_name = model_path+'.tflite'
output_file_name = model_path+'.cc'
command = ["xxd", "-i", input_file_name]

with open(output_file_name, "w") as outfile:
    result = subprocess.run(command, stdout=outfile, text=True)
    if result.returncode == 0:
        print(f"Header file '{output_file_name}' generated successfully.")
    else:
        print("Error generating header file.")

Header file 'lenet5_mnist_og/lenet5_mnist_og.cc' generated successfully.


In [ ]:
command = ["xxd", "-i", model_path+'.tflite']
output_file_name = model_path+'.cc'

with open(output_file_name, "w") as outfile:
    result = subprocess.run(command, stdout=outfile, text=True)

    if result.returncode == 0:
        print(f"Header file '{output_file_name}' generated successfully.")
    else:
        print("Error generating header file.")

# Model Analysis

## Tools

### Make a prediction and plot

In [ ]:
import matplotlib.pyplot as plt

def preprocess(image):
  image = image[None, ...]
  return image

# Select one image and get confidence
image_index = 0
image_correct_class = y_test[image_index]
image = x_test[image_index]
processed_image= preprocess(image)
image_probs = model.predict(processed_image)

plt.figure()
plt.imshow(image * 0.5 + 0.5)  # To change [-1, 1] to [0,1]
plt.title('Correct Classification: {}\nPredicted: {} Confidence: {:.2f}%'.format(image_correct_class,np.argmax(image_probs),np.max(image_probs)*100))
plt.show()

## TF Lite Analyzer
Analyze the model to get details about tensors and operations used in the inference

In [12]:
#model_path='lcnn_fashion_og/lcnn_fashion_og'#0,1,3,4,7,8
tf.lite.experimental.Analyzer.analyze(model_path+".tflite")#tiny0,1,3,4,6,7,10,11,12,13

=== lenet5_mnist_og/lenet5_mnist_og.tflite ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the CONV_2D op takes
tensor #0 and tensor #11 and tensor #10 as input and produces tensor #12 as output.

Subgraph#0 main(T#0) -> [T#24]
  Op#0 CONV_2D(T#0, T#11, T#10[-1165, 9613, 3762, -3414, 1241, ...]) -> [T#12]
  Op#1 TANH(T#12) -> [T#13]
  Op#2 AVERAGE_POOL_2D(T#13) -> [T#14]
  Op#3 CONV_2D(T#14, T#9, T#8[-1906, -2292, -496, 2450, -3509, ...]) -> [T#15]
  Op#4 TANH(T#15) -> [T#16]
  Op#5 AVERAGE_POOL_2D(T#16) -> [T#17]
  Op#6 CONV_2D(T#17, T#7, T#6[-434, 7236, 2772, -3913, 63, ...]) -> [T#18]
  Op#7 TANH(T#18) -> [T#19]
  Op#8 RESHAPE(T#19, T#1[-1, 480]) -> [T#20]
  Op#9 FULLY_CONNECTED(T#20, T#5, T#4[2684, -1280, -1049, 1238, -691, ...]) -> [T#21]
  Op#10 TANH(T#21) -> [T#22]
  Op#11 FULLY_CONNECTED(T#22, T#3, T#2[-1720, 2359, 3133, -963, -2054, ...]) -> [T#23]
  Op#12 SOFTMAX(T#23) -> [T#24]

Tens

## Accuracy

In [13]:
# Load the quantized model
interpreter = tf.lite.Interpreter(model_path + '.tflite')
interpreter.allocate_tensors()

# Get input and output tensor details
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

# Check quantization parameters
input_scale, input_zero_point = input_details["quantization"]
output_scale, output_zero_point = output_details["quantization"]
print(f"Input scale: {input_scale}, Input zero point: {input_zero_point}")
print(f"Output scale: {output_scale}, Output zero point: {output_zero_point}")

# Initialize variables for accuracy calculation
correct_predictions = 0
total_predictions = len(x_test)

# Loop over the entire validation set (one sample at a time)
for i in range(total_predictions):
    # Prepare a single image for the model (reshape to match input size)
    img = x_test[i:i+1]  # Shape (1, 32, 32, 3)
    
    # Quantize the input (scale and zero point adjustment)
    quantized_val = tf.cast(img / input_scale + input_zero_point, tf.int8)

    # Run the model with the quantized input
    interpreter.set_tensor(input_details["index"], quantized_val.numpy())
    interpreter.invoke()

    # Get the quantized logits (output tensor)
    outputs = interpreter.get_tensor(output_details["index"])

    # Reverse the quantization for the logits (convert back to float)
    outputs_float = (tf.cast(outputs, tf.float32) - output_zero_point) * output_scale

    # Apply softmax to get probabilities
    outputs_prob = tf.nn.softmax(outputs_float, axis=-1)

    # Get the predicted class (index of the max probability)
    quantized_pred = tf.argmax(outputs_prob, axis=1)

    # Check if the prediction matches the true label
    if quantized_pred.numpy()[0] == y_test[i]:
        correct_predictions += 1

# Calculate accuracy
accuracy = (correct_predictions / total_predictions) * 100
print(f"Quantized model accuracy on validation set: {accuracy:.02f}%")

Input scale: 0.003921568859368563, Input zero point: -128
Output scale: 0.00390625, Output zero point: -128


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


Quantized model accuracy on validation set: 98.64%


## Tensor Lookup Table

In [26]:
# Ensure model_path is defined
if not os.path.exists(model_path + ".tflite"):
    print(f"Model file not found: {model_path + '.tflite'}")
    exit()

# Initialize the TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=model_path + ".tflite")
interpreter.allocate_tensors()  # Allocate tensors

# Check tensor allocation
tensor_details = interpreter.get_tensor_details()
if not tensor_details:
    print("Tensor details are empty after allocation.")
    exit()
else:
    print(f"Number of tensors allocated: {len(tensor_details)}")

def find_hex_string(file_path, hex_string):
    hex_bytes = bytes.fromhex(hex_string)
    hex_length = len(hex_bytes)
    print(f"The length of the hex string is: {hex_length}")
    
    with open(file_path, 'rb') as file:
        binary_data = file.read()
        position = binary_data.find(hex_bytes)
        print(f"The position of the hex string is: {position}")
        
        return position if position != -1 else None

# Define output file name
output_file_name = model_path + '.cc'  # Ensure model_path is defined

# Start writing to the output file
with open(output_file_name, "a") as outfile:
    outfile.write('uint8_t tensor_lookup_table[] = {\n')  # Corrected type for uint8_t

    print("Kernels and weights:")
    
    # Automatically identify Conv2D and Dense layers
    parameter_tensors = []
    i = 0
    for detail in tensor_details:
        layer_name = detail['name']  # Get the full layer name
        print(f"Checking layer {i}: {layer_name}")  # Debugging output
        if ('Conv2D' in layer_name or 'MatMul' in layer_name) and ('ReadVariableOp' not in layer_name and 'BiasAdd' not in layer_name):
            parameter_tensors.append(detail['index'])
        i+=1

    # Check if parameter_tensors were found
    if not parameter_tensors:
        print("No Conv2D or Dense layers found.")

    for i in parameter_tensors:
        detail = tensor_details[i]
        print(f'Index: {detail["index"]}\t '
              f'Name: {detail["name"]}\t '
              f'Shape: {detail["shape"]}')
        
        # Get the target tensor and convert it to hex
        target_tensor = interpreter.get_tensor(i)
        target_tensor_hex = target_tensor.tobytes().hex()
        print(f'Beginning hex on flatbuffer: {target_tensor_hex[0:32]}')
        
        # Find the position of the hex string in the model file
        position = find_hex_string(model_path + ".tflite", target_tensor_hex)
        
        # Write the tensor details to the output file
        if position is not None:
           outfile.write(f'{{{detail["index"]}, {position}, {len(target_tensor_hex) // 2}}},\n')

    outfile.write('};\n')  # Close the array


Number of tensors allocated: 43
Kernels and weights:
Checking layer 0: serving_default_input_1:0
Checking layer 1: tiny_vgg/flatten_6/Const
Checking layer 2: tiny_vgg/dense_18/BiasAdd/ReadVariableOp
Checking layer 3: tiny_vgg/dense_18/MatMul
Checking layer 4: tiny_vgg/dense_17/BiasAdd/ReadVariableOp
Checking layer 5: tiny_vgg/dense_17/MatMul
Checking layer 6: tiny_vgg/dense_16/BiasAdd/ReadVariableOp
Checking layer 7: tiny_vgg/dense_16/MatMul
Checking layer 8: tiny_vgg/dense_15/BiasAdd/ReadVariableOp
Checking layer 9: tiny_vgg/dense_15/MatMul
Checking layer 10: tiny_vgg/conv2d_35/BiasAdd/ReadVariableOp
Checking layer 11: tiny_vgg/conv2d_35/Conv2D
Checking layer 12: tiny_vgg/conv2d_34/BiasAdd/ReadVariableOp
Checking layer 13: tiny_vgg/conv2d_34/Conv2D
Checking layer 14: tiny_vgg/conv2d_33/BiasAdd/ReadVariableOp
Checking layer 15: tiny_vgg/conv2d_33/Conv2D
Checking layer 16: tiny_vgg/conv2d_32/BiasAdd/ReadVariableOp
Checking layer 17: tiny_vgg/conv2d_32/Conv2D
Checking layer 18: tiny_vgg/

## Save image as an array of bytes

In [41]:
dataset_path = '_datasets/fashion_int8_100'
image = quantized_val[:100]
with open(dataset_path + '.bin', 'wb') as f:
  f.write(bytearray(image))
    
input_file_name = dataset_path + '.bin'
output_file_name = dataset_path + '.h'
command = ["xxd", "-i", input_file_name]

with open(output_file_name, "w") as outfile:
    result = subprocess.run(command, stdout=outfile, text=True)
    if result.returncode == 0:
        print(f"Header file '{output_file_name}' generated successfully.")
    else:
        print("Error generating header file.")

Header file '_datasets/fashion_int8_100.h' generated successfully.


In [42]:
# Flatten the list if necessary (in case y_test contains lists inside)
labels_string = "const int labels[100] = {{{}}}".format(', '.join(str(x) for x in y_test[:100]))
with open(dataset_path+'.h', "a") as f:
    f.write(labels_string)

In [40]:
quantized_val = tf.cast(x_test / input_scale + input_zero_point, tf.int8)
print((quantized_val.shape))
print((x_train.shape))

(10000, 32, 32, 1)
(60000, 32, 32, 1)
